In [68]:
import pyomo.environ as pyo
from pyomo.gdp import Disjunction, Disjunct
import json
import os

In [70]:
json ={
    "jobs": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
    "processing_time": {
        "1": 34, "2": 38, "3": 39, "4": 20, "5": 47, "6": 47, "7": 14, "8": 13,
        "9": 28, "10": 20, "11": 32, "12": 25, "13": 16, "14": 35, "15": 11
    },
    "release_time": {
        "1": 64, "2": 195, "3": 200, "4": 192, "5": 135, "6": 189, "7": 82, "8": 104,
        "9": 123, "10": 29, "11": 43, "12": 49, "13": 156, "14": 11, "15": 52
    },
    "due_time": {
        "1": 302, "2": 461, "3": 473, "4": 332, "5": 464, "6": 518, "7": 180, "8": 195,
        "9": 319, "10": 169, "11": 267, "12": 224, "13": 268, "14": 256, "15": 129
    }
}

In [85]:

m = pyo.ConcreteModel()

m.I = pyo.Set(initialize=json["jobs"])
m.p = pyo.Param(m.I, initialize={int(k): v for k, v in json["processing_time"].items()})
m.r = pyo.Param(m.I, initialize={int(k): v for k, v in json["release_time"].items()})
m.d = pyo.Param(m.I, initialize={int(k): v for k, v in json["due_time"].items()})

# Define the bounds for start times: 
# Each job must start no earlier than its release time and no later than its due date minus processing time.
def x_bounds_rule(m, i):
    return (m.r[i], m.d[i] - m.p[i])
m.x = pyo.Var(m.I, bounds=x_bounds_rule)

# Introduce binary variables:
# y_first[i] = 1 if job i is chosen as the first job.
# y_last[i]  = 1 if job i is chosen as the last job.
m.y_first = pyo.Var(m.I, domain=pyo.Binary)
m.y_last  = pyo.Var(m.I, domain=pyo.Binary)
m.y = pyo.Var(m.I, m.I, domain=pyo.Binary)

# Compute bounds for makespan:
lower_bound_makespan = min(json["release_time"][str(i)] + json["processing_time"][str(i)] for i in json["jobs"])
upper_bound_makespan = max(json["due_time"][str(i)] for i in json["jobs"])
m.makespan = pyo.Var(bounds=(lower_bound_makespan, upper_bound_makespan))


def predecessor_assignment_rule(m, i):
    return sum(m.y[i,j] for j in m.I if j != i) + m.y_first[i] == 1
m.predecessor_assignment = pyo.Constraint(m.I, rule=predecessor_assignment_rule)

def successor_assignment_rule(m, i):
    return  sum(m.y[j,i] for j in m.I if j != i) == 1
m.successor_assignment = pyo.Constraint(m.I, rule=successor_assignment_rule)

def first_job_rule(m):
    return sum(m.y_first[i] for i in m.I) == 1
m.first_job = pyo.Constraint(rule=first_job_rule)

def last_job_rule(m):
    return sum(m.y_last[i] for i in m.I) == 1
m.last_job = pyo.Constraint(rule=last_job_rule)

def demorgan_rule(m, i):
    return m.y_first[i] + m.y_last[i] <= 1
m.demorgan = pyo.Constraint(m.I, rule=demorgan_rule)

def makespan_constraint(m, i):
    return m.x[i] + m.p[i] <= m.makespan
m.makespan_constraint = pyo.Constraint(m.I, rule=makespan_constraint)

m.x_hat = pyo.Var(m.I, m.I, m.I, domain=pyo.NonNegativeReals)
m.x_hat_first = pyo.Var(m.I, m.I, domain=pyo.NonNegativeReals)
m.x_hat_last = pyo.Var(m.I, m.I, domain=pyo.NonNegativeReals)

def linking_rule1(m, i):
    return m.x[i] == sum(m.x_hat[i, i, j] for j in m.I if j != i) + m.x_hat_last[i, i]
m.linking1 = pyo.Constraint(m.I, rule=linking_rule1)

def linking_rule2(m, i):
    return m.x[i] == sum(m.x_hat[i, j, i] for j in m.I if j != i) + m.x_hat_first[i, i]
m.linking2 = pyo.Constraint(m.I, rule=linking_rule2)

def linking_rule3(m, i):
    return m.x[i] == sum(m.x_hat_first[i, j] for j in m.I)
m.linking3 = pyo.Constraint(m.I, rule=linking_rule3)

# def linking_rule4(m, i):
#     return m.x[i] == sum(m.x_hat_last[i, j] for j in m.I)
# m.linking4 = pyo.Constraint(m.I, rule=linking_rule4)

def sequencing_rule(m, i, j):
    if i == j:
        return pyo.Constraint.Skip
    return m.x_hat[j, i, j] - m.x_hat[i, i, j] >= m.p[i] * m.y[i, j]
m.sequencing = pyo.Constraint(m.I, m.I, rule=sequencing_rule)

def x_hat_bounds_rule1(m, i, j):
    if i == j:
        return pyo.Constraint.Skip
    return m.x_hat[j, i, j] <= (m.d[j] - m.p[j]) * m.y[i, j]
m.x_hat_bounds1 = pyo.Constraint(m.I, m.I, rule=x_hat_bounds_rule1)

def x_hat_bounds_rule2(m, i, j):
    if i == j:
        return pyo.Constraint.Skip
    return m.y[i, j] * max(m.r[j], m.r[i]+m.p[i]) <= m.x_hat[j, i, j]

def x_hat_bounds_rule3(m, i, j):
    if i == j:
        return pyo.Constraint.Skip
    return m.x_hat[i, i, j] <= min(m.d[j]-m.p[j], m.d[i]-m.p[i]-m.p[j] ) * m.y[i, j]

# def sequencing_rule_last(m, i, j):
#     if i == j:
#         return pyo.Constraint.Skip
#     return m.x_hat_last[i, i] - m.x_hat_last[j, i] >= m.p[j] * m.y_last[i]
# m.sequencing_last = pyo.Constraint(m.I, m.I, rule=sequencing_rule_last)

def sequencing_rule_first(m, i, j):
    if i == j:
        return pyo.Constraint.Skip
    return m.x_hat_first[i, j] - m.x_hat_first[i, i] >= m.p[i] * m.y_first[i]
m.sequencing_first = pyo.Constraint(m.I, m.I, rule=sequencing_rule_first)

# Define objective: minimize makespan
m.obj = pyo.Objective(expr=m.makespan, sense=pyo.minimize)

# Solve the model
solver = pyo.SolverFactory('gurobi')
results = solver.solve(m, tee=True)

for i in m.I:
    print(f"Job {i}: Start time = {pyo.value(m.x[i])}, Finish time = {pyo.value(m.x[i]) + m.p[i]}")


Set parameter Username
Academic license - for non-commercial use only - expires 2026-01-30
Read LP format model from file /tmp/tmpn0973ymv.pyomo.lp
Reading time = 0.00 seconds
x1: 737 rows, 916 columns, 2925 nonzeros
Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: 12th Gen Intel(R) Core(TM) i7-1265U, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 737 rows, 916 columns and 2925 nonzeros
Model fingerprint: 0x8f755f2e
Variable types: 676 continuous, 240 integer (240 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+02]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 5e+02]
  RHS range        [1e+00, 5e+01]
Presolve removed 246 rows and 462 columns
Presolve time: 0.00s
Presolved: 491 rows, 454 columns, 1559 nonzeros
Variable types: 241 continuous, 213 integer (213 binary)

Root relaxation: infeasible, 123 iterations, 0.00 seconds (0.0

ValueError: No value for uninitialized NumericValue object x[1]

In [82]:

m = pyo.ConcreteModel()

m.I = pyo.Set(initialize=json["jobs"])
m.p = pyo.Param(m.I, initialize={int(k): v for k, v in json["processing_time"].items()})
m.r = pyo.Param(m.I, initialize={int(k): v for k, v in json["release_time"].items()})
m.d = pyo.Param(m.I, initialize={int(k): v for k, v in json["due_time"].items()})

# Define the bounds for start times: 
# Each job must start no earlier than its release time and no later than its due date minus processing time.
def x_bounds_rule(m, i):
    return (m.r[i], m.d[i] - m.p[i])
m.x = pyo.Var(m.I, bounds=x_bounds_rule)

# Introduce binary variables:
# y_first[i] = 1 if job i is chosen as the first job.
# y_last[i]  = 1 if job i is chosen as the last job.
m.y_first = pyo.Var(m.I, domain=pyo.Binary)
m.y_last  = pyo.Var(m.I, domain=pyo.Binary)
m.y = pyo.Var(m.I, m.I, domain=pyo.Binary)

# Compute bounds for makespan:
lower_bound_makespan = min(json["release_time"][str(i)] + json["processing_time"][str(i)] for i in json["jobs"])
upper_bound_makespan = max(json["due_time"][str(i)] for i in json["jobs"])
m.makespan = pyo.Var(bounds=(lower_bound_makespan, upper_bound_makespan))


def M_rule(m, i, j):
    if i == j:
        return pyo.Param.Skip
    else:
        return m.d[i] - m.r[j]
m.M = pyo.Param(m.I, m.I, initialize=M_rule, within=pyo.Any)

def immediate_precedence_rule(m, i, j):
    if i == j:
        return pyo.Constraint.Skip
    else:
        return m.x[i] + m.p[i] <= m.x[j] + m.M[i, j] * (1 - m.y[i, j])
m.immediate_precedence = pyo.Constraint(m.I, m.I, rule=immediate_precedence_rule)

# def first_job_constraint_rule(m, i, j):
#     if i == j:
#         return pyo.Constraint.Skip
#     else:
#         return m.x[i] + m.p[i] <= m.x[j] + m.M[i, j] * (1 - m.y_first[i])
# m.first_job_constraint = pyo.Constraint(m.I, m.I, rule=first_job_constraint_rule)

# def last_job_constraint_rule(m, i, j):
#     if i == j:
#         return pyo.Constraint.Skip
#     else:
#         return m.x[j] + m.p[j] <= m.x[i] + m.M[i, j] * (1 - m.y_last[j])
# m.last_job_constraint = pyo.Constraint(m.I, m.I, rule=last_job_constraint_rule)

def predecessor_assignment_rule(m, i):
    return sum(m.y[i,j] for j in m.I if j != i) + m.y_first[i] == 1
m.predecessor_assignment = pyo.Constraint(m.I, rule=predecessor_assignment_rule)

def successor_assignment_rule(m, i):
    return m.y_last[i] + sum(m.y[j,i] for j in m.I if j != i) == 1
m.successor_assignment = pyo.Constraint(m.I, rule=successor_assignment_rule)

def first_job_rule(m):
    return sum(m.y_first[i] for i in m.I) == 1
m.first_job = pyo.Constraint(rule=first_job_rule)

def last_job_rule(m):
    return sum(m.y_last[i] for i in m.I) == 1
m.last_job = pyo.Constraint(rule=last_job_rule)

def demorgan_rule(m, i):
    return m.y_first[i] + m.y_last[i] <= 1
m.demorgan = pyo.Constraint(m.I, rule=demorgan_rule)

def makespan_constraint(m, i):
    return m.x[i] + m.p[i] <= m.makespan
m.makespan_constraint = pyo.Constraint(m.I, rule=makespan_constraint)

# Define objective: minimize makespan
m.obj = pyo.Objective(expr=m.makespan, sense=pyo.minimize)

# Solve the model
solver = pyo.SolverFactory('gurobi')
results = solver.solve(m, tee=True)

for i in m.I:
    print(f"Job {i}: Start time = {pyo.value(m.x[i])}, Finish time = {pyo.value(m.x[i]) + m.p[i]}")

Set parameter Username
Academic license - for non-commercial use only - expires 2026-01-30
Read LP format model from file /tmp/tmptbdif0h7.pyomo.lp
Reading time = 0.00 seconds
x1: 272 rows, 256 columns, 1169 nonzeros
Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: 12th Gen Intel(R) Core(TM) i7-1265U, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 272 rows, 256 columns and 1169 nonzeros
Model fingerprint: 0xa5c60cf7
Variable types: 16 continuous, 240 integer (240 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+02]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 5e+02]
  RHS range        [1e+00, 5e+02]
Presolve removed 54 rows and 33 columns
Presolve time: 0.02s
Presolved: 218 rows, 223 columns, 947 nonzeros
Variable types: 16 continuous, 207 integer (207 binary)

Root relaxation: objective 2.390000e+02, 79 iterations, 0.00 second

In [ ]:
m = pyo.ConcreteModel()

m.I = pyo.Set(initialize=json["jobs"])
m.p = pyo.Param(m.I, initialize={int(k): v for k, v in json["processing_time"].items()})
m.r = pyo.Param(m.I, initialize={int(k): v for k, v in json["release_time"].items()})
m.d = pyo.Param(m.I, initialize={int(k): v for k, v in json["due_time"].items()})
# Create an index set for ordered pairs (i,j) with i != j.
def pair_filter(i, j):
    return i != j
m.Ipairs = pyo.Set(dimen=2, initialize=m.I * m.I, filter=pair_filter)
# m.Ipairs.pprint()

# Define the bounds for start times: 
# Each job must start no earlier than its release time and no later than its due date minus processing time.
def x_bounds_rule(m, i):
    return (m.r[i], m.d[i] - m.p[i])
m.x = pyo.Var(m.I, bounds=x_bounds_rule)

# Introduce binary variables:
# y_first[i] = 1 if job i is chosen as the first job.
# y_last[i]  = 1 if job i is chosen as the last job.
m.y_first = pyo.Var(m.I, domain=pyo.Binary)
m.y_last  = pyo.Var(m.I, domain=pyo.Binary)
m.y = pyo.Var(m.Ipairs, domain=pyo.Binary)
# m.y.pprint()

# Disjunction-sum constraints:
def pred_disjunction_rule(m, i):
    # For job i acting as predecessor: either it has a follower or it is last.
    return sum(m.y[i, j] for j in m.I if j != i) + m.y_last[i] == 1
m.PredDisc = pyo.Constraint(m.I, rule=pred_disjunction_rule)

def succ_disjunction_rule(m, j):
    # For job j acting as successor: either it has a predecessor or it is first.
    return sum(m.y[i, j] for i in m.I if i != j) + m.y_first[j] == 1
m.SuccDisc =  pyo.Constraint(m.I, rule=succ_disjunction_rule)

# There must be exactly one first and one last job overall.
m.OneFirst =  pyo.Constraint(expr = sum(m.y_first[j] for j in m.I) == 1)
m.OneLast  =  pyo.Constraint(expr = sum(m.y_last[i] for i in m.I) == 1)

def demorgan_rule(m, i):
    return m.y_first[i] + m.y_last[i] <= 1
m.demorgan = pyo.Constraint(m.I, rule=demorgan_rule)

# ----------------------------
# Disaggregated (Convex-Hull) Variables
# ----------------------------
# For each pair (i,j) in m.Ipairs, create a predecessor copy (x_hat_pre)
# for job i when it immediately precedes job j and a successor copy (x_hat_succ)
# for job j in that disjunct.
m.x_hat_pre = pyo.Var(m.Ipairs, domain=pyo.NonNegativeReals)
m.x_hat_succ = pyo.Var(m.Ipairs, domain=pyo.NonNegativeReals)

# For the "first" and "last" alternatives (when no predecessor or successor is chosen)
m.x_hat_first = pyo.Var(m.I, domain=pyo.NonNegativeReals)
m.x_hat_last = pyo.Var(m.I, domain=pyo.NonNegativeReals)

# ----------------------------
# Linking (Re-aggregation) Constraints
# ----------------------------
# Re-aggregate the disaggregated variables to recover the original start time.
def linking_pre_rule(m, i):
    # For job i as predecessor: sum over all disaggregated copies when i is the predecessor plus its "last" alternative.
    return m.x[i] == sum(m.x_hat_pre[i,j] for j in m.I if j != i) + m.x_hat_last[i]
m.linking_pre = pyo.Constraint(m.I, rule=linking_pre_rule)

def linking_succ_rule(m, j):
    # For job j as successor: sum over all disaggregated copies when j is the successor plus its "first" alternative.
    return m.x[j] == sum(m.x_hat_succ[i,j] for i in m.I if i != j) + m.x_hat_first[j]
m.linking_succ = pyo.Constraint(m.I, rule=linking_succ_rule)

# ----------------------------
# Immediate Precedence (Sequencing) Constraints
# ----------------------------
# For each pair (i,j) in m.Ipairs, if job i immediately precedes job j (y[i,j] = 1)
# then the successor copy must be at least the predecessor copy plus processing time of i.
def sequencing_rule(m, i, j):
    return m.x_hat_succ[i,j] - m.x_hat_pre[i,j] >= m.p[i] * m.y[i,j]
m.sequencing = pyo.Constraint(m.Ipairs, rule=sequencing_rule)

# ----------------------------
# Bounds on Disaggregated Variables
# ----------------------------
def x_hat_pre_lb(m, i, j):
    return m.x_hat_pre[i,j] >= m.r[i] * m.y[i,j]
m.x_hat_pre_lb = pyo.Constraint(m.Ipairs, rule=x_hat_pre_lb)

def x_hat_pre_ub(m, i, j):
    return m.x_hat_pre[i,j] <= (m.d[i] - m.p[i]) * m.y[i,j]
m.x_hat_pre_ub = pyo.Constraint(m.Ipairs, rule=x_hat_pre_ub)

# For the successor copy, a common lower bound is r[i] + p[i] (you may refine this further).
def x_hat_succ_lb(m, i, j):
    return m.x_hat_succ[i,j] >= (m.r[i] + m.p[i]) * m.y[i,j]
m.x_hat_succ_lb = pyo.Constraint(m.Ipairs, rule=x_hat_succ_lb)

def x_hat_succ_ub(m, i, j):
    return m.x_hat_succ[i,j] <= (m.d[j] - m.p[j]) * m.y[i,j]
m.x_hat_succ_ub = pyo.Constraint(m.Ipairs, rule=x_hat_succ_ub)

# Bounds for first and last copies:
def x_hat_first_lb(m, j):
    return m.x_hat_first[j] >= m.r[j] * m.y_first[j]
m.x_hat_first_lb = pyo.Constraint(m.I, rule=x_hat_first_lb)

def x_hat_first_ub(m, j):
    return m.x_hat_first[j] <= (m.d[j] - m.p[j]) * m.y_first[j]
m.x_hat_first_ub = pyo.Constraint(m.I, rule=x_hat_first_ub)

def x_hat_last_lb(m, i):
    return m.x_hat_last[i] >= m.r[i] * m.y_last[i]
m.x_hat_last_lb = pyo.Constraint(m.I, rule=x_hat_last_lb)

def x_hat_last_ub(m, i):
    return m.x_hat_last[i] <= (m.d[i] - m.p[i]) * m.y_last[i]
m.x_hat_last_ub = pyo.Constraint(m.I, rule=x_hat_last_ub)

# Compute bounds for makespan:
lower_bound_makespan = min(json["release_time"][str(i)] + json["processing_time"][str(i)] for i in json["jobs"])
upper_bound_makespan = max(json["due_time"][str(i)] for i in json["jobs"])
m.makespan = pyo.Var(bounds=(lower_bound_makespan, upper_bound_makespan))

def makespan_constraint(m, i):
    return m.x[i] + m.p[i] <= m.makespan
m.makespan_constraint = pyo.Constraint(m.I, rule=makespan_constraint)
m.obj = pyo.Objective(expr=m.makespan, sense=pyo.minimize)

# # Solve the model
solver = pyo.SolverFactory('gurobi')
results = solver.solve(m, tee=True)

for i in m.I:
    print(f"Job {i}: Start time = {pyo.value(m.x[i])}, Finish time = {pyo.value(m.x[i]) + m.p[i]}")

Set parameter Username
Academic license - for non-commercial use only - expires 2026-01-30
Read LP format model from file /tmp/tmplbg18si2.pyomo.lp
Reading time = 0.00 seconds
x1: 1202 rows, 706 columns, 3450 nonzeros
Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: 12th Gen Intel(R) Core(TM) i7-1265U, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 1202 rows, 706 columns and 3450 nonzeros
Model fingerprint: 0xd995ba0c
Variable types: 466 continuous, 240 integer (240 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+02]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 5e+02]
  RHS range        [1e+00, 5e+01]
Presolve removed 175 rows and 101 columns
Presolve time: 0.01s
Presolved: 1027 rows, 605 columns, 2934 nonzeros
Variable types: 398 continuous, 207 integer (207 binary)

Root relaxation: objective 2.677218e+02, 1038 iterations, 0

In [41]:
m = pyo.ConcreteModel()

m.I = pyo.Set(initialize=json["jobs"])
m.p = pyo.Param(m.I, initialize={int(k): v for k, v in json["processing_time"].items()})
m.r = pyo.Param(m.I, initialize={int(k): v for k, v in json["release_time"].items()})
m.d = pyo.Param(m.I, initialize={int(k): v for k, v in json["due_time"].items()})

# Immediate precedence Concepts
m.x = pyo.Var(m.I, within=pyo.NonNegativeReals)

# first job disjuncts
def first_job_disjunct_rule(disjunct, i):
    m = disjunct.model()
    disjunct.cons=pyo.ConstraintList()
    # xi+ pi <= xj for all i not equal to j
    for j in m.I:
        if i != j:
            disjunct.cons.add(m.x[i] + m.p[i] <= m.x[j])
m.first_job_disjunct = Disjunct(m.I, rule=first_job_disjunct_rule)

# last job disjuncts
def last_job_disjunct_rule(disjunct, i):
    m = disjunct.model()
    disjunct.cons=pyo.ConstraintList()
    # xj + pj <= xi for all i not
    for j in m.I:
        if i != j:
            disjunct.cons.add(m.x[j] + m.p[j] <= m.x[i])
m.last_job_disjunct = Disjunct(m.I, rule=last_job_disjunct_rule)

def first_job_disjunction_rule(m):
# Return a list of first-job disjuncts, one per job
        return [m.first_job_disjunct[i] for i in m.I]
m.first_job_disjunction = Disjunction(rule=first_job_disjunction_rule)

def last_job_disjunction_rule(m):
# Return a list of last-job disjuncts, one per job
    return [m.last_job_disjunct[i] for i in m.I]
m.last_job_disjunction = Disjunction(rule=last_job_disjunction_rule)

# Logic Expression
def logic_expression_rule(m, i):
    return pyo.lnot(pyo.land(m.first_job_disjunct[i].indicator_var, m.last_job_disjunct[i].indicator_var))
m.logic_expression = pyo.LogicalConstraint(m.I, rule=logic_expression_rule)

# Immediate precedence disjuncts
def immediate_precedence_disjunct_rule(disjunct, i, j):
    m = disjunct.model()
    if i == j:
        disjunct.deactivate() # Deactivate the disjunct if i == j
    else:
        disjunct.cons = pyo.Constraint(expr=m.x[i] + m.p[i] <= m.x[j])
m.immediate_precedence_disjunct = Disjunct(m.I, m.I, rule=immediate_precedence_disjunct_rule)

# Successor Disjunction
def successor_disjunction_rule(m, i):
    return [m.immediate_precedence_disjunct[i, j] for j in m.I if j != i] + [m.last_job_disjunct[i]]
m.SuccessorDisjunction = Disjunction(m.I, rule=successor_disjunction_rule)

# Predecessor Disjunction
def predecessor_disjunction_rule(m, i):
    return [m.immediate_precedence_disjunct[j, i] for j in m.I if j != i] + [m.first_job_disjunct[i]]
m.PredecessorDisjunction = Disjunction(m.I, rule=predecessor_disjunction_rule)